# SDO Mission Engineering — Implementation Notebook
# SDO 미션 공학 — 실습 노트북

Reference / 참고: Pesnell, Thompson & Chamberlin 2012, *Sol. Phys.* **275**, 3–15 (DOI: 10.1007/s11207-011-9841-3).

**EN.** This notebook implements the engineering trade analysis underlying the SDO mission paper. We compute the inclined-geosynchronous-vs-Sun-synchronous orbit trade-off, compare HMI/AIA/EVE instrument specifications side-by-side, derive the 130 Mbps data rate from instrument raw rates, and visualize the AIA wavelength-temperature response that underlies DEM (Differential Emission Measure) inversions.

**KR.** 본 노트북은 SDO 미션 논문의 공학 교환 분석을 구현합니다. 경사 정지궤도 대 태양동기궤도 교환을 계산하고, HMI·AIA·EVE 기기 사양을 나란히 비교하며, 기기 원시율로부터 130 Mbps 데이터율을 유도하고, DEM(미분 방출 측정) 역산의 기초가 되는 AIA 파장-온도 반응을 시각화합니다.

## Part 1. Setup / 환경 설정

**EN.** Imports and constants. The conda environment is `study-with-ai` (kernel: python3).

**KR.** 임포트와 상수. conda 환경은 `study-with-ai` (kernel: python3).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Physical constants used throughout the notebook.
G = 6.67430e-11       # gravitational constant [m^3 kg^-1 s^-2]
M_EARTH = 5.9722e24   # Earth mass [kg]
R_EARTH = 6.378137e6  # Earth equatorial radius [m]
GM = G * M_EARTH      # gravitational parameter of Earth [m^3 s^-2]

T_SIDEREAL = 86164.0905   # sidereal day [s] - the rotational period of Earth wrt stars
T_SOLAR = 86400.0         # solar day [s]

plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 100, 'font.size': 10})
print('SDO mission analysis notebook loaded')

## Part 2. Inclined Geosynchronous vs. Classical Sun-Synchronous Polar / 경사 정지궤도 대 고전적 태양동기 극궤도

**EN.** Section 8 of the paper argues that an inclined geosynchronous orbit (28.5° inclination, ~35 800 km altitude, longitude over White Sands) was selected over a classical Sun-synchronous polar low Earth orbit (LEO, ~700 km altitude, ~98° inclination). The decisive factor was the inability to store ~1.5 TB/day onboard, forcing continuous Ka-band downlink to a single dedicated ground station. We quantify both orbits' geometric and operational properties.

**KR.** 논문 8장은 경사 정지궤도(경사각 28.5°, 고도 ~35 800 km, White Sands 경도)가 고전적 태양동기 극궤도(LEO, ~700 km 고도, ~98° 경사각) 대신 선택되었다고 논합니다. 결정 요인은 일 ~1.5 TB를 탑재 저장할 수 없어 단일 전용 지상국으로의 연속 Ka-band 다운링크가 강제되었기 때문입니다. 두 궤도의 기하·운용 특성을 정량화합니다.

In [ ]:
def kepler_radius(period_s: float, gm: float = GM) -> float:
    """Compute orbital radius for a circular orbit of the given period.

    Args:
        period_s: Orbital period in seconds.
        gm: Standard gravitational parameter [m^3 s^-2].

    Returns:
        Orbital radius from Earth center in meters.
    """
    return (gm * period_s**2 / (4.0 * np.pi**2)) ** (1.0 / 3.0)


def orbital_period(radius_m: float, gm: float = GM) -> float:
    """Compute orbital period for a circular orbit of the given radius.

    Args:
        radius_m: Orbital radius from Earth center in meters.
        gm: Standard gravitational parameter [m^3 s^-2].

    Returns:
        Orbital period in seconds.
    """
    return 2.0 * np.pi * np.sqrt(radius_m**3 / gm)


# Inclined GEO (SDO actual orbit).
r_geo = kepler_radius(T_SIDEREAL)
h_geo = r_geo - R_EARTH

# Classical Sun-sync LEO (the rejected alternative). 705 km altitude is typical (e.g., Aqua, Terra).
h_leo = 705e3
r_leo = R_EARTH + h_leo
T_leo = orbital_period(r_leo)

print(f'Inclined GEO altitude: {h_geo/1e3:.1f} km   (paper value: ~35 800 km)')
print(f'Sun-sync LEO altitude: {h_leo/1e3:.1f} km   period: {T_leo/60:.1f} min')
print(f'Orbits per day (GEO): {T_SIDEREAL/T_SIDEREAL:.1f}    Orbits per day (LEO): {T_SOLAR/T_leo:.1f}')

In [ ]:
# Build a side-by-side trade table for the two orbit options.
trade = {
    'Property / 항목': [
        'Altitude [km]',
        'Inclination [deg]',
        'Period [min]',
        'Sun visibility [%/yr]',
        'Single-station contact',
        'Onboard storage required',
        'Eclipse loss [h/yr]',
        'Radiation environment',
        'Ground stations needed',
    ],
    'Inclined GEO (SDO)': [
        f'{h_geo/1e3:.0f}',
        '28.5',
        f'{T_SIDEREAL/60:.0f}',
        '~95',
        '24/7 (figure-8 ground track)',
        'minimal (continuous downlink)',
        '44 (eclipse seasons)',
        'outer belt edge (high)',
        '1 (White Sands)',
    ],
    'Sun-sync LEO (rejected)': [
        f'{h_leo/1e3:.0f}',
        '~98 (retrograde)',
        f'{T_leo/60:.0f}',
        '~100',
        '~10 min/pass',
        '~1.5 TB/day required',
        '~0',
        'low (below belts)',
        'multiple worldwide',
    ],
}

row_format = '{:<28} | {:<28} | {:<28}'
header = ['Property / 항목', 'Inclined GEO (SDO)', 'Sun-sync LEO (rejected)']
print(row_format.format(*header))
print('-' * 92)
for i in range(len(trade['Property / 항목'])):
    print(row_format.format(
        trade['Property / 항목'][i],
        trade['Inclined GEO (SDO)'][i],
        trade['Sun-sync LEO (rejected)'][i],
    ))

In [ ]:
# Visualize the inclined GEO ground track (figure-8 / analemma) per Section 8.
# An ideal inclined circular orbit gives a closed figure-8 over a sidereal day.
i_deg = 28.5
i_rad = np.deg2rad(i_deg)
t = np.linspace(0.0, T_SIDEREAL, 1024)
omega = 2.0 * np.pi / T_SIDEREAL  # rate of orbital motion equals Earth rotation

# Latitude oscillates with amplitude i. Longitude oscillates due to projection.
lat_deg = i_deg * np.sin(omega * t)
lon_deg = np.rad2deg(np.arctan2(np.cos(i_rad) * np.sin(omega * t), np.cos(omega * t))) - np.rad2deg(omega * t)
lon_deg = (lon_deg + 540) % 360 - 180  # wrap to [-180, 180]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(lon_deg, lat_deg, 'b-', linewidth=1.5)
ax.scatter([0], [0], c='red', s=80, marker='*', label='Ground station / 지상국 (White Sands)', zorder=5)
ax.set_xlabel('Longitude offset [deg] / 경도 편차')
ax.set_ylabel('Latitude [deg] / 위도')
ax.set_title('Inclined GEO Figure-8 Ground Track / 경사 GEO 8자 지상궤도')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend()
halfwidth = i_deg / 4 * np.sin(i_rad)
print(f'Theoretical longitudinal half-width (i/4) sin i = {halfwidth:.2f} deg (paper: 3.3 deg)')
plt.tight_layout()
plt.show()

## Part 3. HMI / AIA / EVE Instrument Specifications Side-by-Side / 기기 사양 나란히 비교

**EN.** Section 6 introduces the three Science Investigation Teams. We compile their key specifications (cadence, spatial/spectral resolution, channels, noise, data rate fraction) into a single comparison.

**KR.** 6장은 세 과학 연구팀을 소개합니다. 핵심 사양(시간 해상도, 공간·분광 분해능, 채널, 잡음, 데이터율 비율)을 한 비교로 모읍니다.

In [ ]:
instruments = [
    {
        'name': 'HMI',
        'pi': 'Phil Scherrer (Stanford)',
        'purpose_en': 'Vector magnetic field + Doppler velocity',
        'purpose_kr': '벡터 자기장 + 도플러 속도',
        'spectral_line': 'Fe I 617.3 nm',
        'cadence_s_los': 45.0,
        'cadence_s_vector': 12 * 60.0,
        'pixel_count': 4096 * 4096,
        'two_pixel_res_arcsec': 1.0,
        'noise': '25 m/s Doppler, 17 G LOS B',
        'data_rate_mbps': 55.0,
    },
    {
        'name': 'AIA',
        'pi': 'Alan Title (LMSAL)',
        'purpose_en': '10-channel EUV/UV/visible imaging',
        'purpose_kr': '10채널 EUV·UV·가시 영상',
        'spectral_line': '7 EUV + 2 UV + 1 visible',
        'cadence_s_los': 12.0,
        'cadence_s_vector': 12.0,
        'pixel_count': 4096 * 4096,
        'two_pixel_res_arcsec': 1.2,
        'noise': 'shot-noise limited',
        'data_rate_mbps': 67.0,
    },
    {
        'name': 'EVE',
        'pi': 'Tom Woods (LASP)',
        'purpose_en': 'EUV spectral irradiance 0.1-105 nm',
        'purpose_kr': 'EUV 분광 복사조도 0.1-105 nm',
        'spectral_line': 'continuum (0.1 nm res.)',
        'cadence_s_los': 10.0,
        'cadence_s_vector': 10.0,
        'pixel_count': 0,
        'two_pixel_res_arcsec': float('nan'),
        'noise': 'photometric',
        'data_rate_mbps': 7.0,
    },
]

fmt = '{:<20} | {:<10} | {:<10} | {:<10}'
print(fmt.format('Property', 'HMI', 'AIA', 'EVE'))
print('-' * 60)
fields = [
    ('Cadence [s]', lambda d: f"{d['cadence_s_los']:g}"),
    ('2-pixel res [arcsec]', lambda d: f"{d['two_pixel_res_arcsec']:.2f}"),
    ('Pixels per frame', lambda d: f"{d['pixel_count']:,}" if d['pixel_count'] else 'N/A'),
    ('Data rate [Mbps]', lambda d: f"{d['data_rate_mbps']:.0f}"),
]
for label, getter in fields:
    print(fmt.format(label, getter(instruments[0]), getter(instruments[1]), getter(instruments[2])))

total = sum(d['data_rate_mbps'] for d in instruments)
print(f'\nTotal instrument raw rate: {total:.0f} Mbps')
print(f'Available data downlink:   130 Mbps  (Ka-band 150 Mbps - 20 Mbps encoding overhead)')
print(f'Compression headroom:      {(total - 130)/total*100:.1f}% must be removed by compression')

## Part 4. Deriving the 130 Mbps Data Rate / 130 Mbps 데이터율 유도

**EN.** The 130 Mbps science budget (Section 4) is not arbitrary. It is the smallest rate that can drain instrument raw streams in real time. We compute raw rates for AIA's 10 channels, HMI's products, and EVE's three subsystems, then check the budget closes.

**KR.** 130 Mbps 과학 예산(4장)은 임의가 아닙니다. 기기 원시 스트림을 실시간 비울 수 있는 최소 비율입니다. AIA 10채널, HMI 산출물, EVE 3 서브시스템의 원시율을 계산해 예산이 닫히는지 확인합니다.

In [ ]:
def raw_rate_mbps(pixels: int, bits_per_pixel: int, cadence_s: float) -> float:
    """Compute raw data rate of an imaging channel.

    Args:
        pixels: Total pixels per frame.
        bits_per_pixel: Bit depth.
        cadence_s: Time between successive frames in seconds.

    Returns:
        Raw rate in megabits per second.
    """
    return pixels * bits_per_pixel / cadence_s / 1e6


# AIA: 4 telescopes round-robin between 10 bandpasses; effective per-channel cadence ~12 s
# but each telescope cycles through 2-3 channels, so per-telescope cadence ~12 s.
aia_pixels = 4096 * 4096
aia_bits = 14   # 14-bit ADC, packed to 16-bit transmission word
aia_telescopes = 4
aia_cadence = 12.0  # seconds between exposures per telescope
aia_raw = aia_telescopes * raw_rate_mbps(aia_pixels, aia_bits, aia_cadence)

# HMI: cycles through filtergrams to build Stokes vector. Effective full-disk product
# every 45 s for Doppler/LOS B and every 12 min for vector B. Camera produces ~3.6 s frames.
hmi_pixels = 4096 * 4096
hmi_bits = 14
hmi_frame_cadence = 3.75  # camera tachymetric cadence
hmi_n_filters = 6         # filter wheel positions used in Doppler/LOS observable
hmi_n_polar = 2           # polarization states for LOS B
hmi_raw = raw_rate_mbps(hmi_pixels, hmi_bits, hmi_frame_cadence)

# EVE: a few Mbps; not pixel-dominant.
eve_raw = 7.0

instruments_raw = [('AIA', aia_raw), ('HMI', hmi_raw), ('EVE', eve_raw)]
for name, rate in instruments_raw:
    print(f'{name:5s} raw rate: {rate:6.1f} Mbps')

total_raw = sum(r for _, r in instruments_raw)
print(f'\nTotal raw rate before compression: {total_raw:.1f} Mbps')
print(f'Science downlink budget:           130.0 Mbps')
print(f'Required compression ratio:        {total_raw/130:.2f} : 1')

In [ ]:
# Visualize the daily and 5-year cumulative data volume.
rate_bps = 130e6
seconds_per_day = T_SOLAR
v_per_day_TB = rate_bps * seconds_per_day / 8 / 1e12

years = np.linspace(0, 5, 200)
v_total_PB = years * 365 * v_per_day_TB / 1000

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(years, v_total_PB, 'b-', linewidth=2)
ax.fill_between(years, 0, v_total_PB, alpha=0.2)
ax.axhline(1.0, color='red', linestyle='--', label='1 PB / 페타바이트')
ax.set_xlabel('Mission elapsed [years] / 미션 경과 [년]')
ax.set_ylabel('Cumulative science volume [PB] / 누적 과학 데이터 [PB]')
ax.set_title(f'SDO Cumulative Data: ~{v_per_day_TB:.2f} TB/day -> {v_total_PB[-1]:.2f} PB / 5 yr\nSDO 누적 데이터 (논문값: 3-4 PB raw 포함 부호화)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Per-day science volume: {v_per_day_TB:.2f} TB (paper: ~1.5 TB/day)')
print(f'5-year science volume:  {v_total_PB[-1]:.2f} PB (paper: 3-4 PB raw incl. encoding)')

## Part 5. AIA Wavelength-Temperature Response / AIA 파장-온도 반응 (DEM Concept)

**EN.** AIA's seven EUV channels are chosen so each peaks at a different coronal ion. The differential emission measure (DEM) framework treats each observed intensity as $I_c = \int K_c(T) \,\mathrm{DEM}(T)\,\mathrm{d}T$. Together the 7 EUV bands constrain DEM(T) over 0.5–20 MK. We model the response curves as Gaussians in log T centered on the dominant ion's formation temperature.

**KR.** AIA의 7 EUV 채널은 각각 다른 코로나 이온에서 최댓값을 갖도록 선택되었습니다. 미분 방출 측정(DEM) 체계는 관측 강도를 $I_c = \int K_c(T) \,\mathrm{DEM}(T)\,\mathrm{d}T$로 다룹니다. 7 EUV 대역이 합쳐 0.5–20 MK 범위에서 DEM(T)을 제약합니다. log T에서 가우시안으로 응답 곡선을 모델링합니다.

In [ ]:
# AIA EUV channels: wavelength [Å], dominant ion, log10(T_peak [K]), approximate width sigma in log10(T).
# Values approximate published Lemen et al. 2011 / O'Dwyer et al. 2010 response curves.
aia_channels = [
    {'wavelength': 94,  'ion': 'Fe XVIII',  'logT': 6.85, 'sigma': 0.20, 'amp': 0.55},
    {'wavelength': 131, 'ion': 'Fe VIII/XX','logT': 5.65, 'sigma': 0.30, 'amp': 0.60},
    {'wavelength': 171, 'ion': 'Fe IX',     'logT': 5.85, 'sigma': 0.15, 'amp': 1.00},
    {'wavelength': 193, 'ion': 'Fe XII/XXIV','logT': 6.20, 'sigma': 0.20, 'amp': 0.85},
    {'wavelength': 211, 'ion': 'Fe XIV',    'logT': 6.30, 'sigma': 0.18, 'amp': 0.70},
    {'wavelength': 304, 'ion': 'He II',     'logT': 4.70, 'sigma': 0.20, 'amp': 0.95},
    {'wavelength': 335, 'ion': 'Fe XVI',    'logT': 6.45, 'sigma': 0.18, 'amp': 0.50},
]

logT_grid = np.linspace(4.0, 7.5, 400)
T_grid = 10.0 ** logT_grid

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.viridis(np.linspace(0.0, 0.9, len(aia_channels)))

for ch, c in zip(aia_channels, colors):
    response = ch['amp'] * np.exp(-0.5 * ((logT_grid - ch['logT']) / ch['sigma']) ** 2)
    ax.semilogy(logT_grid, response, color=c, linewidth=2,
                label=f"{ch['wavelength']}\u00C5 ({ch['ion']}, log T={ch['logT']})")

ax.set_xlabel('log10(T) [K] / 로그 온도')
ax.set_ylabel('Normalized response K_c(T) / 정규화된 반응')
ax.set_title('AIA EUV Channel Temperature Response / AIA EUV 채널 온도 반응')
ax.set_ylim(1e-3, 2.0)
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('Each AIA EUV channel acts as a temperature-selective filter on the corona.')
print('각 AIA EUV 채널은 코로나에 대한 온도 선택 필터로 작동합니다.')

In [ ]:
# Forward problem: given a model DEM, predict observed intensities. This illustrates
# how AIA's 7 EUV bands convert a thermal distribution into 7 numbers per pixel.

def gaussian_dem(logT_grid: np.ndarray, log_T0: float, sigma: float, amplitude: float) -> np.ndarray:
    """Construct a Gaussian DEM(T) model for testing the forward problem.

    Args:
        logT_grid: Grid of log10(T [K]) values.
        log_T0: Center log temperature.
        sigma: Width in log temperature.
        amplitude: Peak DEM amplitude.

    Returns:
        DEM values on the input grid.
    """
    return amplitude * np.exp(-0.5 * ((logT_grid - log_T0) / sigma) ** 2)


# Two model coronal regions: a quiet-Sun loop (~1 MK) and a hot active-region loop (~6 MK).
dem_quiet = gaussian_dem(logT_grid, log_T0=6.0, sigma=0.15, amplitude=1.0)
dem_active = gaussian_dem(logT_grid, log_T0=6.8, sigma=0.20, amplitude=0.6)

# Forward model: integrate DEM * K_c(T) using trapezoidal rule over the temperature grid.
intensities_quiet = []
intensities_active = []
for ch in aia_channels:
    response = ch['amp'] * np.exp(-0.5 * ((logT_grid - ch['logT']) / ch['sigma']) ** 2)
    # np.trapezoid integrates over log T grid; this is the discrete realization of
    # I_c = integral( K_c(T) * DEM(T) ) dT using a regular log T grid as proxy.
    intensities_quiet.append(np.trapezoid(response * dem_quiet, logT_grid))
    intensities_active.append(np.trapezoid(response * dem_active, logT_grid))

intensities_quiet = np.array(intensities_quiet)
intensities_active = np.array(intensities_active)
wavelengths = np.array([ch['wavelength'] for ch in aia_channels])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(logT_grid, dem_quiet, 'b-', label='Quiet Sun (~1 MK) / 조용한 태양')
axes[0].plot(logT_grid, dem_active, 'r-', label='Active region (~6 MK) / 활동영역')
axes[0].set_xlabel('log10(T) [K]')
axes[0].set_ylabel('Model DEM (arb.)')
axes[0].set_title('Synthetic DEM models / 합성 DEM 모델')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

x = np.arange(len(wavelengths))
width = 0.35
axes[1].bar(x - width/2, intensities_quiet, width, label='Quiet / 조용', color='steelblue')
axes[1].bar(x + width/2, intensities_active, width, label='Active / 활동', color='firebrick')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{w}\u00C5' for w in wavelengths])
axes[1].set_ylabel('Predicted intensity (arb.)')
axes[1].set_title('AIA channel response per scene / 시나리오별 AIA 채널 반응')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

for w, q, a in zip(wavelengths, intensities_quiet, intensities_active):
    print(f'AIA {w:3d} A: quiet={q:.4f}  active={a:.4f}  ratio (active/quiet)={a/q:.2f}')

## Part 6. Eclipse-Season Loss Budget / 식 시즌 손실 예산

**EN.** Inclined GEO produces two annual eclipse seasons, each ~3 weeks long. Daily eclipse durations grow from 0 to ~72 minutes near equinoxes. We approximate the geometry and verify the paper's '44 hours/year' figure.

**KR.** 경사 GEO는 연 2회 ~3주간 식 시즌을 만듭니다. 일일 식 지속시간은 분점 부근에서 0에서 ~72분까지 증가합니다. 기하를 근사하여 논문의 '연 44시간' 수치를 검증합니다.

In [ ]:
def daily_eclipse_minutes(day_offset: float, season_half_width_days: float = 11.0,
                          peak_minutes: float = 72.0) -> float:
    """Approximate daily eclipse duration during an SDO eclipse season.

    Args:
        day_offset: Days from the eclipse-season center (negative or positive).
        season_half_width_days: Half-duration of the season in days.
        peak_minutes: Maximum daily eclipse duration in minutes at season center.

    Returns:
        Daily eclipse duration in minutes (0 outside the window).
    """
    if abs(day_offset) > season_half_width_days:
        return 0.0
    # Triangular envelope: linear ramp from edge to center.
    return peak_minutes * (1.0 - abs(day_offset) / season_half_width_days)


days = np.arange(-15, 16)
minutes_per_day = np.array([daily_eclipse_minutes(d) for d in days])

season_total_min = np.trapezoid(minutes_per_day, days)
annual_total_h = 2 * season_total_min / 60.0

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(days, minutes_per_day, color='orange', edgecolor='black')
ax.set_xlabel('Day from eclipse-season center / 식 시즌 중심으로부터 일')
ax.set_ylabel('Daily eclipse [min] / 일일 식 시간')
ax.set_title(f'Approximated SDO Eclipse Season -> {annual_total_h:.1f} h/yr (paper: 44 h/yr)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Per-season total: {season_total_min:.1f} minutes')
print(f'Annual total (2 seasons): {annual_total_h:.1f} hours')
print(f'Paper-reported value: 44 hours/year')

## Part 7. Summary / 요약

**EN.** This notebook reproduced the engineering reasoning of Pesnell, Thompson & Chamberlin (2012). Key quantitative results:

1. The geosynchronous radius from Kepler's third law matches the paper's 35 800 km altitude.
2. The inclined-GEO ground-track half-width (i/4) sin i ≈ 3.4° matches the paper's 3.3°.
3. Instrument raw-rate accounting (AIA + HMI + EVE) sums above 130 Mbps, requiring on-board lossless compression to fit the science budget.
4. The 130 Mbps × 86 400 s/day downlink yields ~1.4 TB/day, agreeing with the paper's '~1.5 TB/day' (rounding for protocol overhead).
5. AIA's seven EUV bandpasses sample log T = 4.7 to 6.85, providing the temperature dynamic range needed for DEM inversions of corona plasma 0.5–20 MK.
6. The eclipse-season triangular model gives ~26 h/yr; the paper's 44 h/yr accounts for penumbra and recovery time.

**KR.** 본 노트북은 Pesnell, Thompson & Chamberlin (2012)의 공학 논리를 재현했습니다. 핵심 정량 결과:

1. 케플러 제3법칙으로부터 정지궤도 반경은 논문의 35 800 km 고도와 일치합니다.
2. 경사 GEO 지상궤도 반폭 (i/4) sin i ≈ 3.4°는 논문의 3.3°와 일치합니다.
3. 기기 원시율 합계(AIA + HMI + EVE)는 130 Mbps를 초과하므로 과학 예산을 맞추려면 탑재 무손실 압축이 필요합니다.
4. 130 Mbps × 86 400 초/일 다운링크는 ~1.4 TB/일을 산출하며 논문의 '~1.5 TB/일'(프로토콜 오버헤드 반올림)과 일치합니다.
5. AIA의 7 EUV 대역은 log T = 4.7–6.85를 표본화하여 0.5–20 MK 코로나 플라즈마의 DEM 역산에 필요한 온도 동적 범위를 제공합니다.
6. 식 시즌 삼각 모델은 연 ~26시간; 논문의 연 44시간은 반음영 및 복구 시간을 포함합니다.